In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import numpy as np

In [7]:
MODEL_NAME = "Fan-s/reddit-tc-bert"
TRAIN_PATH = "/kaggle/input/qna-task2/qna_train.csv"
TEST_PATH = "/kaggle/input/qna-task2/qna_test.csv"
GAMMA = 2.0
SEED = 42

# Set seed
torch.manual_seed(SEED)
np.random.seed(SEED)

# Load data
df = pd.read_csv("/kaggle/input/qna-task2/qna_train.csv")
df_test = pd.read_csv("/kaggle/input/qna-task2/qna_test.csv")

df = df.dropna(subset=["MAIN", "comment_body", "relevance"])
df["MAIN"] = df["MAIN"].astype(str)
df["comment_body"] = df["comment_body"].astype(str)
df["text_pair"] = list(zip(df["MAIN"], df["comment_body"]))

# Train/Val split from train file
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text_pair"].tolist(), df["relevance"].tolist(), test_size=0.1, stratify=df["relevance"], random_state=SEED
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Dataset Class
class RedditDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        topic, comment = self.texts[idx]
        encoding = self.tokenizer(
            topic,
            comment,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {key: val.squeeze() for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Prepare Datasets
train_dataset = RedditDataset(train_texts, train_labels, tokenizer)
val_dataset = RedditDataset(val_texts, val_labels, tokenizer)

# Define Focal Loss
class FocalLossTrainer(Trainer):
    def __init__(self, *args, alpha=0.15, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = FocalLoss(alpha=alpha, gamma=gamma)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Load model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Wrap model with custom loss
# class ModelWithFocalLoss(nn.Module):
#     def __init__(self, model, alpha=0.15, gamma=2.0):
#         super().__init__()
#         self.model = model
#         self.loss_fn = FocalLoss(alpha=alpha, gamma=gamma)

#     def forward(self, input_ids=None, attention_mask=None, labels=None):
#         outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
#         logits = outputs.logits
#         loss = None
#         if labels is not None:
#             loss = self.loss_fn(logits, labels)
#         return {'loss': loss, 'logits': logits}

# wrapped_model = ModelWithFocalLoss(model)

# Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    precision = precision_score(labels, preds)
    recall = recall_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
    }

trainer = FocalLossTrainer(
    model=model,
    args=TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",     # ✅ corrected
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=4,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        save_total_limit=2,
        logging_dir="./logs",
        logging_steps=50,
        report_to="none",  # optional: disable wandb if not using it
    ),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    alpha=0.15,
    gamma=2.0
)
# Train
trainer.train()

# Evaluate
trainer.evaluate()

/tmp/ipykernel_35/2748846027.py:60: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `FocalLossTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.013200,0.011984,0.884007,0.423581,0.782258,0.290419
2,0.012500,0.011850,0.881810,0.436059,0.727273,0.311377
3,0.009900,0.013271,0.873462,0.480144,0.604545,0.398204
4,0.006800,0.016937,0.870826,0.484211,0.584746,0.413174


{'eval_loss': 0.01693704165518284,
 'eval_accuracy': 0.8708260105448155,
 'eval_f1': 0.4842105263157894,
 'eval_precision': 0.5847457627118644,
 'eval_recall': 0.41317365269461076,
 'eval_runtime': 44.2837,
 'eval_samples_per_second': 51.396,
 'eval_steps_per_second': 0.813,
 'epoch': 4.0}

In [8]:
df_test["MAIN"] = df_test["MAIN"].astype(str)
df_test["comment_body"] = df_test["comment_body"].astype(str)
df_test_texts = list(zip(df_test["MAIN"], df_test["comment_body"]))
df_test_labels = df_test["relevance"].tolist()

test_dataset = RedditDataset(df_test_texts, df_test_labels, tokenizer)

predictions = trainer.predict(test_dataset)
test_preds = np.argmax(predictions.predictions, axis=1)

print("Predictions on test set complete.")
test_true_labels = predictions.label_ids
test_accuracy = accuracy_score(test_true_labels, test_preds)
test_f1 = f1_score(test_true_labels, test_preds)
test_precision = precision_score(test_true_labels, test_preds)
test_recall = recall_score(test_true_labels, test_preds)

print("\nTest Set Metrics:")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"F1 Score: {test_f1:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")

Predictions on test set complete.

Test Set Metrics:
Accuracy: 0.8671
F1 Score: 0.4598
Precision: 0.5697
Recall: 0.3854


In [11]:
class RedditDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels  # labels can now be None
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        topic, comment = self.texts[idx]
        encoding = self.tokenizer(
            topic,
            comment,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {key: val.squeeze() for key, val in encoding.items()}
        
        # --- Add this conditional check ---
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        # If self.labels is None, we simply don't add the "labels" key to the item
        return item

df_test_submission = pd.read_csv("/kaggle/input/qna-task2/CRYPTO_QnA_TEST.csv")
df_test_submission["MAIN"] = df_test_submission["MAIN"].astype(str)
df_test_submission["comment_body"] = df_test_submission["comment_body"].astype(str)
test_submission_texts = list(zip(df_test_submission["MAIN"], df_test_submission["comment_body"]))
test_submission_dataset = RedditDataset(test_submission_texts, None, tokenizer)

submission_predictions = trainer.predict(test_submission_dataset)
final_predicted_relevance = np.argmax(submission_predictions.predictions, axis=1)

df_test_submission["relevance"] = final_predicted_relevance

final_submission_df = df_test_submission[["title", "selftext", "MAIN", "comment_body", "relevance"]]
submission_filename = "crypto_test_qna.csv"
final_submission_df.to_csv(submission_filename, index=False)
print(f"\nSubmission file '{submission_filename}' created successfully!")
print(f"Shape of submission file: {final_submission_df.shape}")
print(final_submission_df.head())


Submission file 'crypto_test_qna.csv' created successfully!
Shape of submission file: (6323, 5)
                                               title  \
0  Which exchange to use to see holdings increase...   
1  The end of this year is approaching, how would...   
2                        When do you pull out? (Ha!)   
3  ETH, BTC, ADA, ATOM, ALGO, any other promising...   
4                  Convince me any of this has value   

                                            selftext  \
0  Wazirx doesn't show how much a portfolio has c...   
1  As the end of the year approaches, I am intere...   
2  So I promised my SO that I would only invest a...   
3  These so far are the ones I’ve locked in and c...   
4  I’ve been watching crypto from the outside for...   

                                                MAIN  \
0  Which exchange to use to see holdings increase...   
1  the end of this year is approaching, how would...   
2  when do you pull out? (ha!) so i promised my s...   
3  et

In [12]:
checkpoint_folder = "/kaggle/working/results/checkpoint-1923"

zip_filename = "task2-reddit-tc-bert.zip"

print(f"\nZipping the checkpoint folder '{checkpoint_folder}' into '{zip_filename}'...")
!zip -r {zip_filename} {checkpoint_folder}

from IPython.display import FileLink
print(f"\nClick the link below to download your checkpoints:")
FileLink(zip_filename)


Zipping the checkpoint folder '/kaggle/working/results/checkpoint-1923' into 'task2-reddit-tc-bert.zip'...
  adding: kaggle/working/results/checkpoint-1923/ (stored 0%)
  adding: kaggle/working/results/checkpoint-1923/model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 7%)
  adding: kaggle/working/results/checkpoint-1923/training_args.bin (deflated 51%)
  adding: kaggle/working/results/checkpoint-1923/special_tokens_map.json (deflated 80%)
  adding: kaggle/working/results/checkpoint-1923/tokenizer_config.json (deflated 73%)
  adding: kaggle/working/results/checkpoint-1923/vocab.txt (deflated 53%)
  adding: kaggle/working/results/checkpoint-1923/optimizer.pt (deflated 19%)
  adding: kaggle/working/results/checkpoint-1923/tokenizer.json (deflated 71%)
  adding: kaggle/working/results/checkpoint-1923/trainer_state.json (deflated 75%)
  adding: kaggle/working/results/checkpoint-1923/scheduler.pt (deflated 56%)
  adding: kaggle/working/results/checkpoint-1923/rng_state.pth (deflated 25%)
  adding: kaggle/working/results/checkpoint-1923/config.json (deflated 51%)

Click the link below to download your checkpoints:


/kaggle/working/task2-reddit-tc-bert.zip